In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('Leads_Reales.csv', skiprows=2, encoding='utf-8-sig')
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.head()

,Año,Mes,Total,Ilocalizable,No Contesta,D. Incorrectos,Duplicados,Efectivos,Abiertos,Ventas,Conversion,Efectividad de Leads
0,2024.0,Enero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Febrero,543,108,33,0,1,389,1,12,3.1%,72%
2,NaN,Marzo,178,37,13,0,0,128,1,7,5.5%,72%
3,NaN,Abril,416,79,46,3,0,288,1,8,2.8%,69%
4,NaN,Mayo,422,33,100,17,0,274,5,13,4.7%,65%


In [6]:
# rellenar Año hacia abajo ANTES de tirar filas (si no, se pierde el 2024)
df['Año'] = df['Año'].ffill()

meses_validos = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio',
                  'Agosto','Septiembre','Octubre','Noviembre','Diciembre']
df['Mes'] = df['Mes'].astype(str).str.strip()

# esto tira todo el bloque de resumen/fórmulas que trae el archivo al final
df = df[df['Mes'].isin(meses_validos)].reset_index(drop=True)

# tirar filas sin ningún dato cargado (ej. Enero 2024 viene vacío)
cols_dato = ['Total','Ilocalizable','No Contesta','D. Incorrectos',
             'Duplicados','Efectivos','Abiertos','Ventas']
df = df.dropna(subset=cols_dato, how='all').reset_index(drop=True)

df.shape  # debe darte (28, 12)

(28, 12)

In [7]:
# creamos un diccionario que empareja cada mes con su número: {'Enero':1, 'Febrero':2, ..., 'Diciembre':12}
mapa_meses = {m: i+1 for i, m in enumerate(meses_validos)}

# reemplazamos el texto del mes ("Enero") por su número (1) usando el diccionario,
# y lo convertimos a entero (Int64 admite nulos, a diferencia del int normal)
df['Mes'] = df['Mes'].map(mapa_meses).astype('Int64')

# forzamos Año a número; si algo no se puede convertir, errors='coerce' pone NaN en vez de tronar
df['Año'] = pd.to_numeric(df['Año'], errors='coerce').astype('Int64')

# vemos las primeras filas para checar que Año y Mes ya quedaron como número
df[['Año','Mes']].head()

,Año,Mes
0,2024,2
1,2024,3
2,2024,4
3,2024,5
4,2024,6


In [8]:
# columnas que traen números con comas de miles (ej. "1,409") como texto
cols_numericas = ['Total','Ilocalizable','No Contesta','D. Incorrectos',
                   'Duplicados','Efectivos','Abiertos','Ventas']

for c in cols_numericas:
    # quitamos la coma de miles y espacios sobrantes (ej. " 1,409 " -> "1409")
    df[c] = df[c].astype(str).str.replace(',', '', regex=False).str.strip()
    # convertimos ya el texto limpio a número
    df[c] = pd.to_numeric(df[c], errors='coerce')

df[cols_numericas].dtypes

Total             int64
Ilocalizable      int64
No Contesta       int64
D. Incorrectos    int64
Duplicados        int64
Efectivos         int64
Abiertos          int64
Ventas            int64
dtype: object

In [9]:
#Conversión y Efectividad de Leads
for c in ['Conversion', 'Efectividad de Leads']:
    # quitamos el símbolo % (ej. "3.1%" -> "3.1")
    df[c] = df[c].astype(str).str.replace('%', '', regex=False).str.strip()
    df[c] = pd.to_numeric(df[c], errors='coerce')

df[['Conversion', 'Efectividad de Leads']].head()

,Conversion,Efectividad de Leads
0,3.1,72
1,5.5,72
2,2.8,69
3,4.7,65
4,1.0,64


In [10]:
# tratamos los outliers: en vez de borrar filas (perderíamos meses completos de datos reales),
# "capamos" los valores extremos a los límites del rango intercuartílico (IQR)
cols_para_outliers = ['Total','Ilocalizable','No Contesta','D. Incorrectos',
                       'Duplicados','Efectivos','Abiertos','Ventas',
                       'Conversion','Efectividad de Leads']

for c in cols_para_outliers:
    Q1, Q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    # .clip() sube los valores por debajo del límite inferior y baja los que están arriba del superior
    df[c] = df[c].clip(lower=lim_inf, upper=lim_sup)

df.describe()

,Año,Mes,Total,Ilocalizable,No Contesta,D. Incorrectos,Duplicados,Efectivos,Abiertos,Ventas,Conversion,Efectividad de Leads
count,28.0,28.0,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,2024.785714,6.071429,1126.191964,123.678571,205.285714,19.107143,33.089286,666.500000,250.107143,26.964286,4.142857,61.642857
std,0.738223,3.452627,414.499022,99.463628,143.242764,19.345474,29.891797,191.859999,236.059839,11.083796,1.249720,11.907714
min,2024.0,1.0,271.375000,5.000000,13.000000,0.000000,0.000000,303.000000,1.000000,6.000000,1.000000,39.000000
25%,2024.0,3.0,946.750000,44.250000,103.000000,3.000000,12.000000,564.000000,37.000000,19.250000,3.275000,52.750000
50%,2025.0,5.5,1170.500000,110.000000,185.500000,13.000000,27.000000,704.500000,200.500000,30.000000,4.050000,64.000000
75%,2025.0,9.0,1397.000000,154.250000,278.000000,31.750000,53.000000,738.000000,368.000000,36.250000,5.275000,70.000000
max,2026.0,12.0,1973.000000,319.250000,540.500000,62.000000,114.500000,999.000000,823.000000,41.000000,5.900000,83.000000


In [11]:
df.to_csv('Leads_Reales_Depurado.csv', index=False)